# Projeto Onfly
Automação que realiza coleta, integração e consolidação de dados a partir de uma página web e de uma API pública.


In [1]:
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium import webdriver
import pandas as pd
import requests
import time
import os


# Configuração webdriver -------------------------------------------------------------------------------
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://www.worldometers.info/world-population/population-by-country/"
driver.get(url)

wait = WebDriverWait(driver, 20)


# Acessa primeiro site ------------------------------------------------------------------------
# Espera a tabela aparecer pela CLASS
wait.until(EC.presence_of_element_located(
    (By.CSS_SELECTOR, "table.datatable-table")
))

time.sleep(2)

# Localiza tabela
table = driver.find_element(By.CSS_SELECTOR, "table.datatable-table")

# Cabeçalhos
headers = table.find_elements(By.TAG_NAME, "th")
columns = [h.text for h in headers]

# Linhas
rows = table.find_elements(By.TAG_NAME, "tr")

data = []

for row in rows[1:]:  # pula header
    cols = row.find_elements(By.TAG_NAME, "td")
    row_data = [col.text for col in cols]
    if row_data:
        data.append(row_data)

df = pd.DataFrame(data, columns=columns)
#print(df.head())

driver.quit()
# Trata dados -------------------------------------------------------------------------------

#display(df)
df["Median Age"] = pd.to_numeric(df["Median Age"], errors="coerce")
df = df.sort_values(by="Median Age", ascending=False)
df = df.head(10)

df = df[[
    "Country (or dependency)",
    "Population 2026",
    "Median Age"
]].copy()

# API não reconhece "&" estou transformando em "and"
df["Country (or dependency)"] = df["Country (or dependency)"].str.replace("&", "and")

#display(df)

# Acessa segundo site -------------------------------------------------------------------------------

def consultar_pais(nome_pais):
    url = f"https://restcountries.com/v3.1/name/{nome_pais}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()[0]

        capital = data.get("capital", ["N/A"])[0]

        linguas = ", ".join(data.get("languages", {}).values())

        moedas = ", ".join(
            [moeda.get("name") for moeda in data.get("currencies", {}).values()]
        )

        return capital, linguas, moedas

    except Exception as e:
        return "Erro", "Erro", "Erro"


# Cria novas colunas e mescla -------------------------------------------------------------------------
capitais = []
linguas_lista = []
moedas_lista = []

for pais in df["Country (or dependency)"]:
    capital, linguas, moedas = consultar_pais(pais)
    
    capitais.append(capital)
    linguas_lista.append(linguas)
    moedas_lista.append(moedas)
    
    time.sleep(0.5)  # pequena pausa pra não sobrecarregar API

df["Capital"] = capitais
df["Spoken languages"] = linguas_lista
df["Coins"] = moedas_lista
#display(df)



# trata dados padrão solicitação -----------------------------------------------------------------------------------------------------------
# caso quisesse padrão BR -----------------------------------------------------------------------------------------------
"""
df["Median Age"] = df["Median Age"].astype(float)

df["Median Age"] = df["Median Age"].apply(lambda x: f"{x:,.1f}".replace(",", "X").replace(".", ",").replace("X", "."))
"""
# caso quisesse padrão BR -----------------------------------------------------------------------------------------------


# tirando "," da coluna (como no exemplo do email)
df["Population 2026"] = df["Population 2026"].str.replace(",", "", regex=False)

#Alterando título (como no exemplo do email)
df = df[[
    "Country (or dependency)",
    "Population 2026",
    "Median Age",
    "Capital",
    "Spoken languages",
    "Coins"
]].copy()

df.columns = [
    "País",
    "Qtd habitantes",
    "Média idade",
    "Capital",
    "Linguagens",
    "Moedas"
]


# Salva excel
#caminho = r"C:\Users\Vargas\Desktop\Progamação\Projeto ONFLY\Teste RPA - Mauricio Cunha Vargas Junior.xlsx"
pasta_atual = os.getcwd()
caminho = os.path.join(pasta_atual, "Teste RPA - Mauricio Cunha Vargas Junior.xlsx")

df.to_excel(caminho, index=False)
print("Arquivo salvo com sucesso!")


# Envia via requisição HTTP POST --------------------------------------------------------------------------
url = "https://n8n.dev.onfly.com.br/webhook/b5264095-007e-4c33-a27f-edee2bb6570b"

usuario = "User"
senha = "Password"

with open(caminho, "rb") as f:
    response = requests.post(url,auth=(usuario, senha),files={"file": f})


print("Status:", response.status_code)
print("Resposta:", response.text)
# ---------------------------------------------------------------------------------------------------------------
# fim




Arquivo salvo com sucesso!
Status: 200
Resposta: 
